# QLoRA Fine-Tuning — Qwen 2.5 **7B** Instruct on Egyptian Civil Code

Standalone Colab notebook for the LegalPolicy_LLM project. Trains a LoRA adapter that adapts **Qwen 2.5 7B Instruct** to bilingual (English + Arabic) explanations of Egyptian Civil Code articles, using QLoRA (4-bit base + LoRA adapters).

## What you upload

Two files from your local repo's [data/](../data/) folder:
- `qa_pairs.jsonl`     — training set (chat-format, one JSON object per line)
- `qa_pairs_val.jsonl` — validation set (same format)

Each line must look like:
```json
{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}], "language": "en", "kind": "explanation"}
```

**Where to set the file names**: Step 2 → `TRAIN_JSONL_NAME` / `VAL_JSONL_NAME`. Defaults match the repo, so you usually don't touch them.

## Required Colab setup

1. **Runtime → Change runtime type → GPU**:
   - **A100 (40 GB)** — ideal, fastest
   - **L4 (24 GB)** — recommended default for 7B QLoRA
   - **T4 (16 GB)** — works (auto-tuned smaller profile), but slow
2. *Optional* — Tools → Secrets → `HF_TOKEN` if you want to push the adapter to HuggingFace Hub.

## Outputs (all written to your Google Drive)

- `qlora-qwen2.5-7b-v1/`            — LoRA adapter checkpoints + best model
- `qlora-qwen2.5-7b-v1/training_metrics.png` — loss / perplexity / accuracy / entropy plots
- `qlora-qwen2.5-7b-v1_merged/`     — full merged model (HF format)
- `qlora-qwen2.5-7b-v1_q4_K_M.gguf` — quantized GGUF for Ollama on your laptop
- `qlora-qwen2.5-7b-v1_Modelfile`   — Ollama Modelfile pointing at the GGUF

## Will it run on your RTX 3050 6 GB Laptop after download?

**Yes — with `num_ctx 2048`.** Numbers below; full instructions in Step 10.

| Quant | File size | VRAM @ 2048 ctx | Quality | Fits 3050 6 GB? |
|-------|-----------|-----------------|---------|------------------|
| q4_K_M | ~4.7 GB | ~5.5 GB | best of these | tight, mostly yes (some CPU offload) |
| q4_K_S | ~4.4 GB | ~5.2 GB | very close to q4_K_M | yes |
| q3_K_M | ~3.7 GB | ~4.5 GB | mild quality loss | yes, with comfortable headroom |

We export **q4_K_M** by default and give you the one-liner to also produce q3_K_M as a fallback if 4_K_M paginates to CPU on your laptop.

## Step 1 — Environment & GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU detected — set Runtime → Change runtime type → GPU.'
p = torch.cuda.get_device_properties(0)
GPU_NAME = p.name
GPU_VRAM_GB = p.total_memory / 1e9
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  bf16={torch.cuda.is_bf16_supported()}')
print(f'device={GPU_NAME}  vram={GPU_VRAM_GB:.1f} GB  cc={p.major}.{p.minor}')

In [ ]:
%pip install -q -U \
    "transformers>=4.45" \
    "peft>=0.13" \
    "bitsandbytes>=0.43" \
    "trl>=0.11" \
    "accelerate>=0.34" \
    "datasets>=3.0" \
    "liger-kernel>=0.4" \
    sentencepiece protobuf einops tensorboard matplotlib
print('deps installed')
# Note: recent TRL releases hard-import liger_kernel inside SFTTrainer, so it is
# installed above even though it is nominally optional. If a later TRL still
# breaks on import, pin it: %pip install -q "trl==0.12.2" and restart the runtime.

## Step 2 — Configuration

All knobs in one place. The training-hyperparameters block auto-tunes for the detected GPU; override anything you want.

**Where to point at your data**: edit `TRAIN_JSONL_NAME` / `VAL_JSONL_NAME` if your files are renamed. The actual upload happens in Step 3.

In [ ]:
from pathlib import Path
import os, random, json

# ============================================================
# Paths & data file names
# ============================================================
PROJECT_ROOT      = Path('/content/legalpolicy')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_JSONL_NAME  = 'qa_pairs.jsonl'        # ← change here if your file is renamed
VAL_JSONL_NAME    = 'qa_pairs_val.jsonl'    # ← change here if your file is renamed
TRAIN_JSONL       = PROJECT_ROOT / TRAIN_JSONL_NAME
VAL_JSONL         = PROJECT_ROOT / VAL_JSONL_NAME
ADAPTER_NAME      = 'qlora-qwen2.5-7b-v1'

# ============================================================
# Model
# ============================================================
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'

# ============================================================
# Training — auto-tuned to the detected GPU. Edit inside the
# matching branch to override.
# ============================================================
if GPU_VRAM_GB >= 35:                                      # A100 / H100
    PROFILE         = 'a100'
    MAX_SEQ_LEN     = 2048
    LORA_R          = 32
    LORA_ALPHA      = 64
    PER_DEV_BATCH   = 4
    GRAD_ACCUM      = 8         # effective batch = 32
elif GPU_VRAM_GB >= 22:                                    # L4 / A10 / 3090 / 4090
    PROFILE         = 'l4'
    MAX_SEQ_LEN     = 2048
    LORA_R          = 32
    LORA_ALPHA      = 64
    PER_DEV_BATCH   = 2
    GRAD_ACCUM      = 16        # effective batch = 32
else:                                                      # T4 16 GB — tight
    PROFILE         = 't4'
    MAX_SEQ_LEN     = 1280
    LORA_R          = 16
    LORA_ALPHA      = 32
    PER_DEV_BATCH   = 1
    GRAD_ACCUM      = 32        # effective batch = 32

LORA_DROPOUT    = 0.1
LR              = 2e-5          # 7B is more sensitive than 3B — keep LR low
EPOCHS          = 3
WARMUP_RATIO    = 0.06
EARLY_STOP_PAT  = 5
EVAL_STEPS      = 50
SEED            = 13

# ============================================================
# Drive — checkpoints, adapter, plots, GGUF all land here
# ============================================================
USE_DRIVE  = True
DRIVE_DIR  = '/content/drive/MyDrive/legalpolicy_qlora'

random.seed(SEED)
print(f'GPU profile: {PROFILE}  ({GPU_NAME}, {GPU_VRAM_GB:.1f} GB)')
print(f'Model: {BASE_MODEL}')
print(f'LoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}')
print(f'Batch: per_dev={PER_DEV_BATCH} × grad_accum={GRAD_ACCUM} → effective={PER_DEV_BATCH*GRAD_ACCUM}')
print(f'Seq len: {MAX_SEQ_LEN} | LR: {LR} | Epochs: {EPOCHS}')
print(f'Adapter dir name: {ADAPTER_NAME}')
print(f'Expected uploads: {TRAIN_JSONL_NAME}, {VAL_JSONL_NAME}')

## Step 3 — Upload your `qa_pairs.jsonl` and `qa_pairs_val.jsonl`

Pick **one** source. Defaults to local upload.

- `DATA_SOURCE = 'upload'` — file picker; select **both** files from your local `data/` folder in one go.
- `DATA_SOURCE = 'drive'`  — read both files from `DRIVE_DATA_DIR` (set below).
- `DATA_SOURCE = 'github'` — git-clone your repo (works only if it's public, or you provide a token).

In [ ]:
DATA_SOURCE    = 'upload'   # 'upload' | 'drive' | 'github'

DRIVE_DATA_DIR = '/content/drive/MyDrive/legalpolicy_data'           # used when DATA_SOURCE='drive'
GITHUB_REPO    = 'https://github.com/ayanasser/LegalPolicy_LLM.git'  # used when DATA_SOURCE='github'
GITHUB_BRANCH  = 'feature/finetuning-PEFT'

if DATA_SOURCE == 'upload':
    from google.colab import files
    print(f'Select BOTH {TRAIN_JSONL_NAME} AND {VAL_JSONL_NAME} from your local data/ folder …')
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(VAL_JSONL_NAME):
            Path(name).rename(VAL_JSONL)
        elif name.endswith(TRAIN_JSONL_NAME):
            Path(name).rename(TRAIN_JSONL)

elif DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    src = Path(DRIVE_DATA_DIR)
    !cp "{src}/{TRAIN_JSONL_NAME}" "{TRAIN_JSONL}"
    !cp "{src}/{VAL_JSONL_NAME}"   "{VAL_JSONL}"

elif DATA_SOURCE == 'github':
    !git clone --depth=1 --branch {GITHUB_BRANCH} {GITHUB_REPO} /content/repo
    repo_data = Path('/content/repo/data')
    !cp "{repo_data}/{TRAIN_JSONL_NAME}" "{TRAIN_JSONL}"
    !cp "{repo_data}/{VAL_JSONL_NAME}"   "{VAL_JSONL}"

else:
    raise ValueError(f'Unknown DATA_SOURCE: {DATA_SOURCE}')

assert TRAIN_JSONL.exists(), f'Missing {TRAIN_JSONL}'
assert VAL_JSONL.exists(),   f'Missing {VAL_JSONL}'
n_train = sum(1 for _ in open(TRAIN_JSONL, encoding='utf-8'))
n_val   = sum(1 for _ in open(VAL_JSONL,   encoding='utf-8'))
print(f'\nDataset ready:')
print(f'  train: {n_train:>4} rows  →  {TRAIN_JSONL} ({TRAIN_JSONL.stat().st_size/1024:.1f} KB)')
print(f'  val:   {n_val:>4} rows  →  {VAL_JSONL}   ({VAL_JSONL.stat().st_size/1024:.1f} KB)')

# Quick schema sanity check
with open(TRAIN_JSONL, encoding='utf-8') as f:
    sample = json.loads(f.readline())
assert 'messages' in sample and isinstance(sample['messages'], list), 'Each line must have a "messages" list.'
print(f'\nFirst row preview: roles={[m["role"] for m in sample["messages"]]}, language={sample.get("language")}')

## Step 4 — Tokenize using Qwen ChatML template

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

ds = load_dataset('json', data_files={'train': str(TRAIN_JSONL), 'validation': str(VAL_JSONL)})

def format_example(ex):
    text = tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    return {'text': text}

ds = ds.map(format_example, remove_columns=[c for c in ds['train'].column_names if c != 'messages'])
print(ds)
print('\n--- SAMPLE (first 1200 chars) ---')
print(ds['train'][0]['text'][:1200])

## Step 5 — Load Qwen 2.5 7B in 4-bit and attach LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='sdpa',
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 6 — Train with QLoRA + custom metrics (accuracy, entropy)

On every eval step we compute, in addition to `eval_loss`:
- **`eval_token_accuracy`** — next-token argmax accuracy on non-padding tokens
- **`eval_avg_entropy`**    — mean predictive entropy (nats) on non-padding tokens; lower = more confident model

Perplexity is derived in the plotting cell as `exp(eval_loss)`.

In [ ]:
import numpy as np, inspect
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    output_dir = f'{DRIVE_DIR}/{ADAPTER_NAME}'
else:
    output_dir = str(PROJECT_ROOT / ADAPTER_NAME)
Path(output_dir).mkdir(parents=True, exist_ok=True)

def preprocess_logits_for_metrics(logits, labels):
    # Reduce per-batch BEFORE accumulation to keep eval memory cheap.
    if isinstance(logits, tuple):
        logits = logits[0]
    log_probs = torch.log_softmax(logits, dim=-1)
    probs     = log_probs.exp()
    entropy   = -(probs * log_probs).sum(dim=-1)        # (B, T)
    pred_ids  = logits.argmax(dim=-1).float()           # (B, T)
    return torch.stack([pred_ids, entropy], dim=-1)     # (B, T, 2)

def compute_metrics(eval_pred):
    preds  = eval_pred.predictions       # (N, T, 2)
    labels = eval_pred.label_ids         # (N, T)
    pred_ids = preds[..., 0].astype(np.int64)
    entropy  = preds[..., 1].astype(np.float32)
    # Causal LM shift: position t predicts t+1
    shift_preds   = pred_ids[..., :-1]
    shift_entropy = entropy[..., :-1]
    shift_labels  = labels[..., 1:]
    mask  = shift_labels != -100
    n     = max(int(mask.sum()), 1)
    correct = (shift_preds == shift_labels) & mask
    return {
        'token_accuracy': float(correct.sum() / n),
        'avg_entropy':    float((shift_entropy * mask).sum() / n),
    }

# Build SFTConfig kwargs in a version-aware way:
# - TRL <=0.11: max_seq_length, dataset_text_field, packing on config
# - TRL >=0.12: max_length replaces max_seq_length
# - transformers 5.x: save_strategy must match eval_strategy when
#   load_best_model_at_end=True, so we use steps for both.
sft_kwargs = dict(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEV_BATCH,
    per_device_eval_batch_size=PER_DEV_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit',
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to=['tensorboard'],
    logging_dir=f'{output_dir}/runs',
    seed=SEED,
)
sft_params = inspect.signature(SFTConfig.__init__).parameters
if 'max_seq_length' in sft_params:
    sft_kwargs['max_seq_length'] = MAX_SEQ_LEN
elif 'max_length' in sft_params:
    sft_kwargs['max_length'] = MAX_SEQ_LEN
if 'dataset_text_field' in sft_params:
    sft_kwargs['dataset_text_field'] = 'text'
if 'packing' in sft_params:
    sft_kwargs['packing'] = False

sft_config = SFTConfig(**sft_kwargs)

trainer_kwargs = dict(
    model=model,
    args=sft_config,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PAT)],
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)
# Newer TRL renamed `tokenizer` → `processing_class` on SFTTrainer.
trainer_params = inspect.signature(SFTTrainer.__init__).parameters
if 'processing_class' in trainer_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
print('trainer ready')
print(f'  TRL accepted: max_seq={"max_seq_length" in sft_params or "max_length" in sft_params}, '
      f'dataset_text_field={"dataset_text_field" in sft_params}, packing={"packing" in sft_params}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {output_dir}/runs

In [ ]:
trainer.train()
trainer.save_model(output_dir)
print(f'Adapter saved to {output_dir}')

## Step 7 — Plot training/eval metrics

Saves a single PNG with four panels:
1. Train vs. eval **loss**
2. Eval **perplexity** (= `exp(eval_loss)`)
3. Eval **token accuracy**
4. Eval **average entropy**

Plus a JSON dump of the full log history for downstream reporting. Both land in your Drive next to the adapter.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, json

logs = trainer.state.log_history
(Path(output_dir) / 'log_history.json').write_text(json.dumps(logs, indent=2))

train_steps, train_losses, train_grad = [], [], []
eval_steps,  eval_losses,  eval_acc, eval_ent = [], [], [], []
for entry in logs:
    if 'loss' in entry and 'eval_loss' not in entry:
        train_steps.append(entry['step'])
        train_losses.append(entry['loss'])
        if 'grad_norm' in entry:
            train_grad.append((entry['step'], entry['grad_norm']))
    if 'eval_loss' in entry:
        eval_steps.append(entry['step'])
        eval_losses.append(entry['eval_loss'])
        if 'eval_token_accuracy' in entry: eval_acc.append(entry['eval_token_accuracy'])
        if 'eval_avg_entropy'    in entry: eval_ent.append(entry['eval_avg_entropy'])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.plot(train_steps, train_losses, label='train', alpha=0.7)
if eval_losses:
    ax.plot(eval_steps, eval_losses, label='eval', marker='o', color='tab:orange')
ax.set(title='Loss', xlabel='step', ylabel='loss'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
if eval_losses:
    ppl = np.exp(np.clip(np.array(eval_losses), 0, 20))
    ax.plot(eval_steps, ppl, marker='o', color='tab:purple')
ax.set(title='Perplexity (eval) — exp(eval_loss)', xlabel='step', ylabel='ppl'); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
if eval_acc:
    ax.plot(eval_steps, eval_acc, marker='o', color='tab:green')
ax.set(title='Token accuracy (eval, next-token argmax on non-pad)', xlabel='step', ylabel='accuracy'); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

ax = axes[1, 1]
if eval_ent:
    ax.plot(eval_steps, eval_ent, marker='o', color='tab:red')
ax.set(title='Average predictive entropy (eval, nats)', xlabel='step', ylabel='entropy'); ax.grid(True, alpha=0.3)

plt.suptitle(f'{ADAPTER_NAME} — training metrics ({PROFILE} profile, {GPU_NAME})', y=1.02)
plt.tight_layout()
out = Path(output_dir) / 'training_metrics.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

# Summary table — final values
print('\n=== Final eval metrics ===')
if eval_losses: print(f'  eval_loss       : {eval_losses[-1]:.4f}')
if eval_losses: print(f'  perplexity      : {np.exp(min(eval_losses[-1], 20)):.2f}')
if eval_acc:    print(f'  token_accuracy  : {eval_acc[-1]:.4f}  ({eval_acc[-1]*100:.1f}%)')
if eval_ent:    print(f'  avg_entropy     : {eval_ent[-1]:.4f} nats')
if train_grad:  print(f'  last grad_norm  : {train_grad[-1][1]:.4f}')

## Step 8 — Sample generations (sanity check)

In [ ]:
model.eval()

def chat(user_msg, max_new_tokens=400):
    msgs = [{'role': 'user', 'content': user_msg}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            temperature=None, top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_ids[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

TESTS = [
    'Explain Article 1 of the Egyptian Civil Code in plain language.',
    'اشرح المادة الأولى من القانون المدني المصري بلغة بسيطة.',
    'Should I sue my employer? Give me a step-by-step strategy.',
]
for q in TESTS:
    print('=== Q:', q)
    print(chat(q))
    print()

## Step 9 — (Optional) Push the LoRA adapter to HuggingFace Hub

In [ ]:
print('Adapter contents:')
for p in sorted(Path(output_dir).iterdir()):
    size = p.stat().st_size if p.is_file() else None
    print(f'  {p.name}  {size if size is not None else "(dir)"}')

PUSH_TO_HUB = False
HF_REPO_ID  = '<your-username>/legalpolicy-qwen2.5-7b-qlora'   # EDIT before flipping the flag
if PUSH_TO_HUB:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = os.environ.get('HF_TOKEN')
    if token: login(token=token)
    else:     login()
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f'Pushed to {HF_REPO_ID}')

## Step 10 — Merge LoRA → full model → GGUF q4_K_M (saved to Drive)

**Default ON.** Produces these files inside `DRIVE_DIR`:
- `<ADAPTER_NAME>_merged/`        — full HF-format model with LoRA folded in
- `<ADAPTER_NAME>_fp16.gguf`      — fp16 GGUF intermediate (large; you can delete after quantize)
- `<ADAPTER_NAME>_q4_K_M.gguf`    — **the file you'll download to your laptop**
- `<ADAPTER_NAME>_Modelfile`      — Ollama Modelfile referencing the q4_K_M

Set `EXPORT_Q3_K_M = True` if you also want a smaller q3_K_M as a fallback for the 3050.

In [ ]:
MERGE_AND_EXPORT = True
EXPORT_Q3_K_M    = False    # set True to also produce a smaller fallback quant

if MERGE_AND_EXPORT:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM

    # Free GPU memory before loading the fp16 base on CPU
    del trainer, model
    import gc; gc.collect(); torch.cuda.empty_cache()

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, device_map='cpu', trust_remote_code=True,
    )
    merged = PeftModel.from_pretrained(base, output_dir).merge_and_unload()
    merged_dir = f'{output_dir}_merged'
    merged.save_pretrained(merged_dir, safe_serialization=True)
    tokenizer.save_pretrained(merged_dir)
    print('Merged →', merged_dir)

    !git clone --depth=1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
    %pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

    gguf_fp16 = f'{output_dir}_fp16.gguf'
    gguf_q4   = f'{output_dir}_q4_K_M.gguf'
    !python /content/llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile {gguf_fp16} --outtype f16
    !cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF && cmake --build build --target llama-quantize -j
    !/content/llama.cpp/build/bin/llama-quantize {gguf_fp16} {gguf_q4} q4_K_M
    print('GGUF q4_K_M →', gguf_q4)

    if EXPORT_Q3_K_M:
        gguf_q3 = f'{output_dir}_q3_K_M.gguf'
        !/content/llama.cpp/build/bin/llama-quantize {gguf_fp16} {gguf_q3} q3_K_M
        print('GGUF q3_K_M →', gguf_q3)

    modelfile_text = f'''FROM {Path(gguf_q4).name}
TEMPLATE """<|im_start|>system
{{{{ .System }}}}<|im_end|>
<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
"""
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.3
PARAMETER num_ctx 2048
SYSTEM "You are a careful explainer of the Egyptian Civil Code. Provide plain-language explanations grounded in the cited articles. Always include a one-line disclaimer that you are not providing legal advice."
'''
    Path(f'{output_dir}_Modelfile').write_text(modelfile_text)
    print('Modelfile →', f'{output_dir}_Modelfile')

    # Disk usage summary
    print('\n=== Drive artifacts (download these to your laptop) ===')
    for path in [gguf_q4, f'{output_dir}_Modelfile']:
        sz = Path(path).stat().st_size / 1e9
        print(f'  {path}   ({sz:.2f} GB)' if sz >= 0.1 else f'  {path}   ({sz*1024:.1f} MB)')

## Step 11 — Run the model on your RTX 3050 6 GB Laptop with Ollama

**Short answer: yes, it runs.** Qwen 2.5 7B at q4_K_M is ~4.7 GB on disk and ~5.5 GB VRAM at `num_ctx 2048`. On a 3050 6 GB Laptop you'll have ~5 GB usable VRAM after the OS — Ollama will fit most of the model on GPU and offload a few layers to CPU. Expect **~10–18 tok/s**.

If it's too tight, re-run Step 10 with `EXPORT_Q3_K_M = True` and use the q3_K_M file instead (~3.7 GB on disk, ~4.5 GB VRAM, very minor quality loss).

### One-time setup on your laptop

1. Install Ollama from https://ollama.com/download.
2. From your Drive, download these two files into a single folder, e.g. `~/legalpolicy/`:
   - `qlora-qwen2.5-7b-v1_q4_K_M.gguf`
   - `qlora-qwen2.5-7b-v1_Modelfile`
3. In that folder, run:
   ```bash
   ollama create legalpolicy-qwen7b -f qlora-qwen2.5-7b-v1_Modelfile
   ollama run  legalpolicy-qwen7b
   ```

### Knobs to tune on the 3050

| Symptom | Fix |
|---------|-----|
| OOM at start | Edit Modelfile: `PARAMETER num_ctx 1024` (halve KV cache) |
| Very slow (<3 tok/s) | Most layers spilled to CPU. Switch to q3_K_M, or shorten context |
| `Out of memory` mid-generation | Cap output: `PARAMETER num_predict 512` |
| Want max throughput | `OLLAMA_NUM_GPU=99 ollama run …` to push everything to GPU; reduce `num_ctx` until it fits |

Sanity-test prompts:
```
>>> Explain Article 1 of the Egyptian Civil Code in plain language.
>>> اشرح المادة الأولى من القانون المدني المصري بلغة بسيطة.
>>> Should I sue my employer? Give me a step-by-step strategy.
```

Then run the project's evaluation harness ([scripts/run_judge_inference.py](../scripts/run_judge_inference.py)) base-vs-tuned to produce the ship/no-ship report.